---
# Curso: Data Mining — Semana 12 - Lunes
**Semana 12 — Miércoles**

**Docente:** Soledad Espezúa Llerena

---

# <font color=green>Laboratorio: Clustering aplicado al espacio</font>

### <font color=green>Repaso</font>

- Un clustering **por atributos** responde: *¿qué observaciones se parecen?*
- Un clustering **espacial** (DBSCAN con coordenadas) responde: *¿dónde se concentran las observaciones?* Luego miramos en el mapa **dónde** quedó cada grupo.
- Para DBSCAN espacial, `eps` debe estar en **metros**: primero reproyectamos.
- K-means y el jerárquico **asignan todos los puntos** a un grupo; DBSCAN deja los puntos aislados como **ruido**. ¿Cuál conviene? depende de la forma de los datos.
- Un **mapa de calor** muestra densidad continua; **no** asigna grupos ni detecta ruido.
- Los patrones dependen de los parámetros (`k`, `eps`, `min_samples`, ancho de banda) y de **dónde hay datos**.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.cluster.hierarchy import to_tree

# Estilo de seaborn para todos los gráficos.
# Los mapas los dibuja GeoPandas (.plot); seaborn da el estilo y los colores.
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## <font color=green>0. Preparamos los archivos</font>

Suban al panel **Archivos** de Colab (ícono de carpeta, a la izquierda) estos dos archivos:

- `weather-stations.csv`: estaciones meteorológicas de Canadá (Environment Canada).
- `canada_provincias.geojson`: límites de las provincias de Canadá (Natural Earth, dominio público). Lo usaremos como capa de fondo.

# <font color=blue>PARTE I. Los datos: estaciones meteorológicas de Canadá</font>

Cada fila es una estación. Usaremos estas columnas:

| Columna | Significado |
|---|---|
| `Stn_Name` | Nombre de la estación |
| `Lat`, `Long` | Latitud y longitud (grados) |
| `Prov` | Provincia |
| `Tm` | Temperatura media del mes (°C) |
| `Tx` | Temperatura máxima más alta del mes (°C) |
| `Tn` | Temperatura mínima más baja del mes (°C) |

**Fuente**:
Notebook del laboratorio de IBM “Density-based Clustering” (copia en CoCalc)
Monthly Climate Summaries – Environment and Climate Change Canada
DBSCAN algorithm: complete guide and application with Python (Towards Data Science).

## <font color=green>1. Cargamos y limpiamos los datos</font>

In [ ]:
df = pd.read_csv("weather-stations.csv")
print("Estaciones en el archivo:", len(df))

# Nos quedamos con las estaciones que tienen las tres temperaturas.
df = df.dropna(subset=["Tm", "Tx", "Tn"]).reset_index(drop=True)
print("Estaciones con Tm, Tx y Tn:", len(df))

display(df[["Stn_Name", "Prov", "Lat", "Long", "Tm", "Tx", "Tn"]].head())

## <font color=green>2. Convertimos la tabla en un GeoDataFrame</font>

Igual que el lunes: `X = longitud`, `Y = latitud`, y `EPSG:4326` porque las coordenadas están en grados.

In [ ]:
#gdf son los datos (puntos): Las estaciones meteorológicas
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Long"], df["Lat"]),  # X = longitud, Y = latitud
    crs="EPSG:4326"
)

# Capa de fondo: provincias de Canadá:es el mapa de fondo (polígonos).
# El GeoJSON es un archivo geoespacial: trae su propia columna geometry
#                                       con los polígonos y su CRS.
provincias = gpd.read_file("canada_provincias.geojson")

print("CRS estaciones :", gdf.crs)
print("CRS provincias :", provincias.crs)

In [ ]:
# Mapa con dos capas: provincias (fondo) + estaciones (puntos).
ax = provincias.plot(figsize=(10, 7), facecolor="white", edgecolor="gray")
gdf.plot(ax=ax, color="black", markersize=4)

ax.set_title("Estaciones meteorológicas de Canadá")
ax.set_axis_off()
plt.show()

### Observación

Las estaciones **no están repartidas de forma uniforme**: se concentran en el sur, donde vive la mayor parte de la población. Recuerden esto al interpretar cualquier patrón espacial.

# <font color=blue>PARTE II. Clustering por atributos y su lectura en el mapa</font>

Pregunta: **¿qué estaciones tienen un clima parecido?**

Agrupamos usando **solo las temperaturas** (`Tm`, `Tx`, `Tn`). La ubicación **no** entra al modelo: la usamos después, para ver en el mapa dónde quedó cada grupo.

## <font color=green>3. K-means con las temperaturas</font>

 k no se elige a ojo. Hay que evaluarlo, como en la unidad de clustering: con el método del codo, silueta y davies_bouldin.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score


# 1) Variables del clustering: solo atributos (no coordenadas).
X_temp = df[["Tm", "Tx", "Tn"]]

# 2) Escalamos: K-means usa distancias, así que todas las variables deben pesar lo mismo.
X_temp_esc = StandardScaler().fit_transform(X_temp)

#-------------------Evaluación-------------------------------
# ¿Cuántos grupos? Probamos k = 2 a 8 con tres criterios.
filas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_temp_esc)
    filas.append({"k": k,
                  "inercia": km.inertia_,                                        # codo
                  "silueta": silhouette_score(X_temp_esc, km.labels_),           # más alto = mejor
                  "davies_bouldin": davies_bouldin_score(X_temp_esc, km.labels_)})  # más bajo = mejor
evaluacion = pd.DataFrame(filas)
display(evaluacion.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, titulo in zip(axes, ["inercia", "silueta", "davies_bouldin"],
                           ["Inercia (codo)", "Silueta (más alto = mejor)", "Davies-Bouldin (más bajo = mejor)"]):
    sns.lineplot(data=evaluacion, x="k", y=col, marker="o", ax=ax)
    ax.axvline(4, color="gray", linestyle="--")   # el k elegido
    ax.set_title(titulo)
plt.show()
#-----------------------------------------------------------------

# 3) K-means con k = 4, el valor que respaldan las métricas.
# random_state fija el resultado.
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
df["cluster"] = kmeans.fit_predict(X_temp_esc)

print(df["cluster"].value_counts().sort_index())

## <font color=green>4. Le ponemos nombre a cada grupo</font>

Los números que asigna K-means (0, 1, 2, 3) son arbitrarios. Para interpretar, ordenamos los grupos por su temperatura media y les damos un nombre.

In [ ]:
# Temperatura media de cada grupo, de menor a mayor.
medias = df.groupby("cluster")["Tm"].mean().sort_values()
print(medias.round(1))

# Asignamos un nombre según ese orden.
nombres = ["1. Muy frío", "2. Frío", "3. Moderado", "4. Templado"]
df["grupo_clima"] = df["cluster"].map(dict(zip(medias.index, nombres)))

# Perfil de cada grupo: promedio de cada variable.
display(
    df.groupby("grupo_clima")[["Tm", "Tx", "Tn", "Long","Lat"]].mean().round(1)
      .assign(estaciones=df["grupo_clima"].value_counts())
)

In [ ]:
# Distribución de la temperatura media por grupo (seaborn).
ax = sns.boxplot(
    data=df.sort_values("grupo_clima"),
    x="grupo_clima", y="Tm",
    hue="grupo_clima", palette="coolwarm", legend=False
)
ax.set_title("Temperatura media por grupo")
ax.set_xlabel("")
ax.set_ylabel("Tm (°C)")
plt.show()

## <font color=green>5. Incorporamos la etiqueta del clúster al mapa</font>

Copiamos la columna `grupo_clima` al GeoDataFrame y coloreamos los puntos por grupo.

In [ ]:
#print("Columnas antes:", list(gdf.columns))

# Copiamos la etiqueta del clúster como una columna nueva del GeoDataFrame
gdf["grupo_clima"] = df["grupo_clima"]

#print("Columnas después:", list(gdf.columns))   # grupo_clima aparece al final
#gdf[["Stn_Name", "Tm", "grupo_clima", "geometry"]].head()


# Un color por grupo, en el mismo orden que los nombres: 1. Muy frío → 4. Templado
#colores = ["#3B4CC0", "#AAC7FD", "#F7B89C", "#B40426"]

ax = provincias.plot(figsize=(10, 7), facecolor="white", edgecolor="gray")
gdf.plot(
    ax=ax,
    column="grupo_clima",   # el color depende del grupo
    categorical=True,       # grupos, no números continuos
    cmap="coolwarm",        # azul = frío, rojo = templado
    #cmap=ListedColormap(colores),   # nuestros colores, en lugar de "coolwarm"
    markersize=8,
    legend=True,
    legend_kwds={"loc": "center left", "bbox_to_anchor": (1, 0.5), "title": "Grupo (por temperatura)"}
)

# Etiqueta de cada provincia: su código postal (AB, BC, ON...) en un punto dentro del polígono.
# representative_point() siempre cae dentro de la provincia (el centroide a veces cae fuera).
for _, prov in provincias.iterrows():
    punto = prov.geometry.representative_point()
    ax.annotate(prov["postal"], xy=(punto.x, punto.y),   #prov["name"]
                ha="center", fontsize=9, fontweight="bold", color="dimgray",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))  # fondo para leerla sobre los puntos

ax.set_title("Clustering por atributos, visto en el mapa")
ax.set_axis_off()
plt.show()

### Interpretación

- La ubicación **no** se usó para formar los grupos, pero en el mapa los grupos forman **regiones**:
  - *Templado*: casi todas sus estaciones están en Columbia Británica, junto al Pacífico.
  - *Moderado*: sobre todo Alberta y el interior de Columbia Británica.
  - *Frío*: el centro y el este del país (Quebec, Ontario, las praderas).
  - *Muy frío*: el norte, principalmente Nunavut.
- Eso es la **interpretación espacial posterior**: el patrón geográfico *emerge* de los atributos.
- La tabla de abajo lo confirma con números: cuántas estaciones de cada grupo hay en cada provincia.

In [ ]:
# ¿En qué provincias cae cada grupo? (tabla de conteos)
pd.crosstab(df["Prov"], df["grupo_clima"])

# Tabla de contingencia (cross tabulation): cruza dos columnas categóricas y
# cuenta cuántas filas hay en cada combinación.
# Aquí las filas son las provincias, las columnas son los grupos de clima,
# y cada celda dice cuántas estaciones de esa provincia cayeron en ese grupo.

# <font color=blue>PARTE III. DBSCAN espacial</font>

Ahora la pregunta es otra: **¿dónde se concentran las estaciones?**

Usamos **solo la ubicación**. DBSCAN forma un grupo cuando hay al menos `min_samples` estaciones a menos de `eps` de distancia; las estaciones aisladas quedan como **ruido** (etiqueta `-1`).

## <font color=green>6. Reproyectamos a metros</font>

`eps` es una distancia, así que necesitamos coordenadas en **metros**.

Canadá es demasiado grande para una sola zona UTM (cada zona cubre 6° de longitud y Canadá abarca más de 80°). Para un país grande se usa una proyección pensada para todo el país: **EPSG:3978** (*Canada Atlas Lambert*), en metros.

In [ ]:
gdf_m = gdf.to_crs("EPSG:3978") #EPSG:3978 Es una proyección cónica de Lambert diseñada para todo Canadá
print("CRS:", gdf_m.crs.name)

# DBSCAN recibe una matriz con dos columnas: X e Y (en metros).
coords_m = np.column_stack([gdf_m.geometry.x, gdf_m.geometry.y]) #pone esas dos listas lado a lado como columnas y forma una matriz.

coords_m[:3]

## <font color=green>7. DBSCAN con eps = 100 km</font>

### ¿Qué valor de eps? El gráfico de k-distancias

Igual que con `k` en K-means, `eps` debe justificarse. Para cada estación medimos la distancia a su **10.º vecino más cercano** (el mismo número que `min_samples`) y ordenamos esas distancias de menor a mayor.

- Donde la curva sube despacio están las estaciones con vecinos cerca (zonas densas).
- Donde la curva **se dispara** (el "codo") están las estaciones aisladas.

El codo sugiere un **rango** razonable para `eps`.

In [ ]:
# Distancia de cada estación a su 10.º vecino más cercano (el mismo número que min_samples).
# n_neighbors=10 incluye a la propia estación, igual que DBSCAN la cuenta en min_samples.
vecinos = NearestNeighbors(n_neighbors=10).fit(coords_m)
distancias, _ = vecinos.kneighbors(coords_m)
k_dist_km = np.sort(distancias[:, -1]) / 1000   # ordenadas de menor a mayor, en km

print("Mediana:", round(np.median(k_dist_km)), "km")
print("Percentil 90:", round(np.percentile(k_dist_km, 90)), "km")

ax = sns.lineplot(x=range(len(k_dist_km)), y=k_dist_km)
ax.axhline(100, color="gray", linestyle="--", label="eps = 100 km (nuestra elección)")
ax.set_title("Gráfico de k-distancias: el codo sugiere un rango para eps")
ax.set_xlabel("Estaciones ordenadas")
ax.set_ylabel("Distancia al 10.º vecino (km)")
ax.legend()
plt.show()

In [ ]:
# ¿Qué pasa con distintos eps por debajo y en el codo? (mismo min_samples = 10)
filas = []
for eps_km in [50, 100, 150, 200, 250]:
    etiquetas = DBSCAN(eps=eps_km * 1000, min_samples=10).fit_predict(coords_m)
    tamanos = pd.Series(etiquetas[etiquetas != -1]).value_counts()   # estaciones por grupo
    filas.append({"eps_km": eps_km,
                  "grupos": len(tamanos),
                  "ruido": (etiquetas == -1).sum(),
                  "grupo_mas_grande": tamanos.max()})
pd.DataFrame(filas)

**Lectura:**

- La mitad de las estaciones tiene 10 vecinos a menos de ~90 km. La curva se dispara cerca de **250 km**: ahí están las estaciones aisladas del norte.
- El codo **no es el mejor eps, sino el máximo razonable**. Con `eps = 250 km` casi todas las estaciones del sur (1 113) se unen en **un solo grupo** de BC a Quebec: solo se separan las aisladas.
- Por eso elegimos **por debajo del codo**, según la **escala de la pregunta**:

| eps | Grupos | Ruido | Qué se ve |
|---|---|---|---|
| 100 km | 7 | 414 | concentraciones regionales |
| 150 km | 3 | 244 | solo oeste, este y Terranova |
| 250 km | 3 | 113 | casi todo el sur en un grupo |

Usamos **100 km** porque queremos ver concentraciones regionales.

**¿Por qué no usamos la silueta, como con K-means?** La silueta solo evalúa los puntos agrupados e ignora el ruido. Con `eps = 50 km` sale "excelente" (0.68), pero descarta el **71 %** de las estaciones como ruido: la métrica premia tirar datos. Por eso en DBSCAN se usa el gráfico de k-distancias (que da el máximo razonable) más el criterio del problema, y se revisa la sensibilidad (sección 9).

In [ ]:
# eps en metros: 100 km = 100_000 m.
# min_samples = 10: se necesitan al menos 10 estaciones cercanas para formar un grupo.
dbscan = DBSCAN(eps=100_000, min_samples=10)
gdf_m["cluster_espacial"] = dbscan.fit_predict(coords_m)

n_grupos = gdf_m["cluster_espacial"].nunique() - (1 if -1 in gdf_m["cluster_espacial"].values else 0)
n_ruido = (gdf_m["cluster_espacial"] == -1).sum()

print("Grupos encontrados:", n_grupos)
print("Estaciones como ruido:", n_ruido, f"({n_ruido / len(gdf_m):.0%})")
print(gdf_m["cluster_espacial"].value_counts().sort_index())

In [ ]:
# Mapa en metros: provincias + ruido (gris) + grupos (colores).
provincias_m = provincias.to_crs("EPSG:3978")   # misma proyección que los puntos

ruido = gdf_m[gdf_m["cluster_espacial"] == -1]
grupos = gdf_m[gdf_m["cluster_espacial"] != -1]

ax = provincias_m.plot(figsize=(10, 7), facecolor="white", edgecolor="gray")
ruido.plot(ax=ax, color="lightgray", markersize=4)   # gris = ruido
grupos.plot(
    ax=ax,
    column="cluster_espacial",
    categorical=True,
    cmap="tab10",
    markersize=10,
    legend=True,
    legend_kwds={"loc": "center left", "bbox_to_anchor": (1, 0.5), "title": "Grupo espacial"}
)

# Etiqueta de cada provincia
for _, prov in provincias_m.iterrows():
    punto = prov.geometry.representative_point()
    ax.annotate(prov["postal"], xy=(punto.x, punto.y),
                ha="center", fontsize=9, fontweight="bold", color="dimgray",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))  # fondo para leerla sobre los puntos


ax.set_title("DBSCAN espacial (eps = 100 km, min_samples = 10) · gris = ruido")
ax.set_axis_off()
plt.show()

### Interpretación

- Los grupos coinciden con las zonas **con más estaciones**: el sur de Columbia Británica, Alberta, el corredor de Ontario y Quebec, y las provincias marítimas.
- El norte queda casi todo como **ruido**: allí las estaciones están a cientos de kilómetros entre sí.
- DBSCAN encontró **concentraciones de estaciones**, no regiones climáticas. Es otra pregunta que la de la Parte II.

## <font color=green>8. ¿Qué pasa si usamos grados en vez de metros?</font>

In [ ]:
# Mismo eps = 100_000, pero con longitud/latitud en grados.
coords_grados = np.column_stack([gdf.geometry.x, gdf.geometry.y])
etiquetas_grados = DBSCAN(eps=100_000, min_samples=10).fit_predict(coords_grados)

print("Grupos con coordenadas en grados:", len(set(etiquetas_grados)))
# Resultado: 1 solo grupo. En grados, 100 000 es más que la vuelta al mundo,
# así que todas las estaciones quedan "cerca" de todas.

Y si usáramos `eps = 1` grado, tampoco sería una distancia fija: en el sur de Canadá (49° N) un grado de longitud mide unos 73 km, pero en el Ártico (70° N) solo unos 38 km.

**Conclusión:** para DBSCAN espacial, primero reproyectamos a metros y luego elegimos `eps` en metros.

## <font color=green>9. Los parámetros cambian el resultado</font>

In [ ]:
# Probamos tres valores de eps con el mismo min_samples.
valores_eps_km = [50, 100, 150]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, eps_km in zip(axes, valores_eps_km):
    etiquetas = DBSCAN(eps=eps_km * 1000, min_samples=10).fit_predict(coords_m)
    n_g = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
    n_r = (etiquetas == -1).sum()

    provincias_m.plot(ax=ax, facecolor="white", edgecolor="gray", linewidth=0.5)
    ax.scatter(coords_m[etiquetas == -1, 0], coords_m[etiquetas == -1, 1], s=3, color="lightgray")
    ax.scatter(coords_m[etiquetas != -1, 0], coords_m[etiquetas != -1, 1], s=6,
               c=etiquetas[etiquetas != -1], cmap="tab10")
    ax.set_title(f"eps = {eps_km} km → {n_g} grupos, {n_r} ruido")
    ax.set_axis_off()

plt.show()

- `eps` pequeño → muchos grupos pequeños y mucho ruido.
- `eps` grande → pocos grupos grandes que unen zonas distintas.

No hay un `eps` "correcto" universal: se elige según la escala del problema (¿100 km tiene sentido para lo que queremos analizar?).

# <font color=blue>PARTE IV. K-means, jerárquico y DBSCAN: ¿cuál conviene?</font>

Los tres algoritmos usan **las columnas que les damos**. La diferencia está en **cómo forman los grupos**:

- **K-means** y **jerárquico** (`AgglomerativeClustering`): ustedes fijan el número de grupos y **todos** los puntos quedan asignados a alguno.
- **DBSCAN**: encuentra el número de grupos, admite formas irregulares y deja los puntos aislados como **ruido**.

Probamos los tres con las mismas columnas, primero con las coordenadas y luego con las temperaturas.

## <font color=green>10. Con las coordenadas: ¿dónde se concentran las estaciones?</font>

### El dendrograma: ¿dónde cortar el árbol?

El clustering jerárquico no pide `k` al inicio: construye un **árbol** (dendrograma) que va uniendo estaciones y grupos de abajo hacia arriba.

- La **altura** de cada unión indica qué tan distintos eran los grupos que se unieron.
- Se corta el árbol con una **línea horizontal** donde hay un **salto grande** de altura: ahí se estaban uniendo grupos muy distintos.
- El número de ramas que corta la línea es el número de grupos.

Primero, el árbol cuando agrupamos por **ubicación** (coordenadas en metros).

In [ ]:
# Árbol con las coordenadas en metros (ward = mismo método que AgglomerativeClustering)
Z_coords = linkage(coords_m, method="ward")

# Etiqueta de cada rama: provincia más frecuente y cuántas estaciones tiene
_, nodos = to_tree(Z_coords, rd=True)
def etiqueta(id_nodo):
    miembros = nodos[id_nodo].pre_order()                         # estaciones de esa rama
    provincia = df["Prov"].iloc[miembros].value_counts().index[0]  # provincia más frecuente
    return f"{provincia} ({len(miembros)})"

corte = 11_000_000   # altura elegida mirando el gráfico (en el eje: 1.1 × 1e7)

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z_coords, truncate_mode="lastp", p=30,   # últimas 30 uniones
           color_threshold=corte,                    # un color por rama bajo la línea
           leaf_label_func=etiqueta, leaf_rotation=90, ax=ax)
ax.axhline(corte, color="gray", linestyle="--")
ax.set_title("Dendrograma con las coordenadas (ward)")
plt.show()

# ¿Cuántos grupos deja ese corte?
grupos_arbol = fcluster(Z_coords, t=corte, criterion="distance")
print("Grupos con ese corte:", len(set(grupos_arbol)))

**Lectura:** las etiquetas indican la provincia más frecuente de cada rama y cuántas estaciones tiene.

- El salto más grande separa **2 ramas**: el **este** (Ontario, Quebec y las provincias del Atlántico) y el **oeste junto con el norte** (BC, Alberta, las praderas y los territorios).
- Hay otro salto claro en **4 ramas**: el este, el **norte** (Yukón, Territorios del Noroeste, Nunavut), las **praderas** (Saskatchewan, Manitoba) y el **oeste** (BC, Alberta).
- En **7** el árbol **no** muestra un salto especial: separa además Ontario de Quebec y BC de Alberta.

Usamos 7 grupos solo para comparar K-means y jerárquico **con el mismo número de grupos que encontró DBSCAN**. Es una decisión para la comparación, no la que sugiere el árbol. Si el objetivo fuera solo el jerárquico, cortaríamos en 2 o en 4.

In [ ]:
# Mismas coordenadas en metros para los tres modelos.
# Usamos 7 grupos en K-means y jerárquico para compararlos con los 7 de DBSCAN.
modelos = {
    "K-means (k=7)": KMeans(n_clusters=7, n_init=10, random_state=42),
    "Jerárquico (7 grupos)": AgglomerativeClustering(n_clusters=7, linkage="ward"),
    "DBSCAN (eps=100 km)": DBSCAN(eps=100_000, min_samples=10),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (nombre, modelo) in zip(axes, modelos.items()):
    etiquetas = modelo.fit_predict(coords_m)
    es_ruido = etiquetas == -1          # solo DBSCAN produce -1

    provincias_m.plot(ax=ax, facecolor="white", edgecolor="gray", linewidth=0.5)
    ax.scatter(coords_m[es_ruido, 0], coords_m[es_ruido, 1], s=4, color="lightgray")
    ax.scatter(coords_m[~es_ruido, 0], coords_m[~es_ruido, 1], s=6,
               c=etiquetas[~es_ruido], cmap="tab10")
    ax.set_title(f"{nombre}\n{len(set(etiquetas[~es_ruido]))} grupos · {es_ruido.sum()} ruido")
    ax.set_axis_off()

plt.show()

**Aquí gana DBSCAN.**

- K-means y el jerárquico reparten **todas** las estaciones, incluidas las aisladas del norte: forman grupos que abarcan territorios enormes y casi vacíos.
- DBSCAN encuentra las zonas densas (con la forma que tengan) y deja el norte como ruido.

## <font color=green>11. Con las temperaturas: ¿qué estaciones tienen clima parecido?</font>

### El dendrograma con las temperaturas

Ahora el mismo árbol, pero agrupando por **clima** (temperaturas escaladas de la Parte II).

In [ ]:
# Árbol con las temperaturas escaladas
Z_temp = linkage(X_temp_esc, method="ward")

# Misma etiqueta: provincia más frecuente de cada rama (la provincia NO entra al modelo)
_, nodos = to_tree(Z_temp, rd=True)

corte = 18   # altura elegida mirando el gráfico: en medio del último salto grande

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z_temp, truncate_mode="lastp", p=30,   # últimas 30 uniones
           color_threshold=corte,                  # un color por rama bajo la línea
           leaf_label_func=etiqueta, leaf_rotation=90, ax=ax)
ax.axhline(corte, color="gray", linestyle="--")
ax.set_title("Dendrograma con las temperaturas (ward)")
plt.show()

# ¿Cuántos grupos deja ese corte?
grupos_arbol = fcluster(Z_temp, t=corte, criterion="distance")
print("Grupos con ese corte:", len(set(grupos_arbol)))

**Lectura:** el último **salto grande** del árbol está entre las alturas ~24 y ~13: ahí las ramas son largas y no hay uniones. Una línea en `corte = 18` cruza **4 ramas** → 4 grupos.

Las etiquetas muestran qué hay en cada rama, aunque la provincia **no** entró al modelo:

- **Nunavut** → muy frío (el Ártico).
- **Ontario, Quebec, Manitoba, Saskatchewan** → frío.
- **Columbia Británica** → templado (costa del Pacífico).
- **Alberta y el Atlántico** (Nueva Escocia, Terranova) → moderado.

Prueben mover la línea: con `corte = 30` salen 3 grupos; con `corte = 10`, muchos más.

El dendrograma confirma lo que dijeron el codo y Davies-Bouldin en la Parte II: **4 grupos**.

In [ ]:
# Mismas temperaturas escaladas de la Parte II para los tres modelos.
# Ojo: aquí eps no está en metros sino en unidades estandarizadas.
modelos_temp = {
    "K-means (k=4)": KMeans(n_clusters=4, n_init=10, random_state=42),
    "Jerárquico (4 grupos)": AgglomerativeClustering(n_clusters=4, linkage="ward"),
    "DBSCAN (eps=0.25)": DBSCAN(eps=0.25, min_samples=10),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (nombre, modelo) in zip(axes, modelos_temp.items()):
    etiquetas = modelo.fit_predict(X_temp_esc)
    es_ruido = etiquetas == -1

    provincias_m.plot(ax=ax, facecolor="white", edgecolor="gray", linewidth=0.5)
    ax.scatter(coords_m[es_ruido, 0], coords_m[es_ruido, 1], s=4, color="lightgray")
    ax.scatter(coords_m[~es_ruido, 0], coords_m[~es_ruido, 1], s=6,
               c=etiquetas[~es_ruido], cmap="tab10")
    ax.set_title(f"{nombre}\n{len(set(etiquetas[~es_ruido]))} grupos · {es_ruido.sum()} ruido")
    ax.set_axis_off()

plt.show()

In [ ]:
# ¿Qué hay dentro del grupo más grande de DBSCAN?
etiquetas_db = DBSCAN(eps=0.25, min_samples=10).fit_predict(X_temp_esc)
df["db_temp"] = etiquetas_db
df.groupby("db_temp")["Tm"].agg(["count", "min", "mean", "max"]).round(1)

**Aquí ganan K-means y el jerárquico.**

- Las temperaturas cambian de forma **continua** de sur a norte: no hay "huecos" entre climas.
- DBSCAN encadena estaciones vecinas en temperatura y termina con un grupo enorme que va de unos −29 °C a +10 °C. Prueben otros valores: con `eps=0.2` aparece mucho más ruido y con `eps=0.4` todo queda en un solo grupo.
- K-means y el jerárquico dividen ese gradiente en grupos útiles (y entre ellos salen parecidos).

| | K-means | Jerárquico | DBSCAN |
|---|---|---|---|
| Número de grupos | Lo fijan ustedes | Lo fijan ustedes (o cortan el dendrograma) | Lo encuentra el algoritmo |
| ¿Asigna todos los puntos? | Sí | Sí | No: marca ruido (-1) |
| Forma de los grupos | Compactos, de tamaño parecido | Depende del enlace | Cualquier forma |
| Funciona mejor cuando… | se quiere segmentar un continuo | además interesa la jerarquía | hay zonas densas separadas por zonas vacías |

**Conclusión:** DBSCAN no "gana" siempre. Gana con las **ubicaciones**, porque los puntos forman concentraciones separadas por vacíos.

# <font color=blue>PARTE V. Mapa de calor vs. clustering espacial</font>

Un **mapa de calor** (estimación de densidad) colorea el mapa según cuántas estaciones hay alrededor de cada lugar. Lo hacemos con `sns.kdeplot`.

## <font color=green>12. Mapa de calor de las estaciones</font>

In [ ]:
ax = provincias_m.plot(figsize=(10, 7), facecolor="white", edgecolor="gray")

sns.kdeplot(
    x=gdf_m.geometry.x,
    y=gdf_m.geometry.y,
    fill=True,          # superficie coloreada
    cmap="rocket_r",    # más oscuro = más denso
    levels=15,          # número de niveles de color
    thresh=0.05,        # no pinta las zonas casi vacías
    alpha=0.8,
    ax=ax
)
gdf_m.plot(ax=ax, color="black", markersize=1)

ax.set_title("Mapa de calor: densidad de estaciones")
ax.set_axis_off()
plt.show()

## <font color=green>13. Comparamos lado a lado</font>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Izquierda: mapa de calor
provincias_m.plot(ax=ax1, facecolor="white", edgecolor="gray", linewidth=0.5)
sns.kdeplot(x=gdf_m.geometry.x, y=gdf_m.geometry.y, fill=True, cmap="rocket_r",
            levels=15, thresh=0.05, alpha=0.8, ax=ax1)
ax1.set_title("Mapa de calor: ¿cuánta densidad hay en cada lugar?")
ax1.set_axis_off()

# Derecha: DBSCAN
provincias_m.plot(ax=ax2, facecolor="white", edgecolor="gray", linewidth=0.5)
ruido.plot(ax=ax2, color="lightgray", markersize=4)
grupos.plot(ax=ax2, column="cluster_espacial", categorical=True, cmap="tab10", markersize=10)
ax2.set_title("DBSCAN: ¿a qué grupo pertenece cada estación?")
ax2.set_axis_off()

plt.show()

| | Mapa de calor | Clustering espacial (DBSCAN) |
|---|---|---|
| ¿Qué produce? | Una **superficie continua** de densidad | Una **etiqueta por punto** |
| ¿Define grupos? | No: muestra "más" o "menos" | Sí: grupo 0, 1, 2…, y ruido (-1) |
| ¿Detecta ruido? | No | Sí: puntos aislados = -1 |
| Parámetro clave | Ancho de banda (`bw_adjust`) | `eps` y `min_samples` |
| Sirve para… | Ver dónde se concentra algo | Usar el grupo como variable en un análisis |
| Cuidado con… | "Pinta" también zonas sin datos (el mar, fuera del país) | Grupos muy sensibles a `eps` |

Los dos dependen de un parámetro de **escala**: el ancho de banda del mapa de calor cumple un papel parecido al `eps` de DBSCAN.

In [ ]:
# El mapa de calor también cambia con su parámetro: bw_adjust (ancho de banda).
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, bw in zip(axes, [0.3, 1.5]):
    provincias_m.plot(ax=ax, facecolor="white", edgecolor="gray", linewidth=0.5)
    sns.kdeplot(x=gdf_m.geometry.x, y=gdf_m.geometry.y, fill=True, cmap="rocket_r",
                levels=15, thresh=0.05, alpha=0.8, bw_adjust=bw, ax=ax)
    ax.set_title(f"bw_adjust = {bw}")
    ax.set_axis_off()
plt.show()

# <font color=blue>PARTE VI. Interpretación y limitaciones</font>

Antes de sacar conclusiones de un mapa con clústeres, revisen:

1. **¿Dónde hay datos?** Los grupos espaciales y el mapa de calor reflejan dónde se instalaron estaciones (cerca de la población), no necesariamente dónde ocurre el fenómeno.
2. **Los parámetros importan.** `k`, `eps`, `min_samples` y `bw_adjust` cambian el resultado. Prueben varios valores antes de concluir.
3. **¿Qué variables entraron al modelo?** Un grupo por atributos y un grupo espacial responden preguntas distintas. No los mezclen sin pensarlo.
4. **Unidades y proyección.** Con coordenadas en grados, `eps` no es una distancia física.
5. **Un mapa muestra un patrón, no su causa.** Que el grupo templado esté en la costa del Pacífico sugiere una explicación (la influencia del océano), pero el clustering no la demuestra.

# <font color=blue>DESAFÍO</font>

## <font color=green>A. Clustering por atributos</font>

Repitan la Parte II con `n_clusters=3`. ¿Qué grupos se fusionan? ¿Siguen formando regiones en el mapa?

## <font color=green>B. DBSCAN espacial</font>

(Si ya lo hicieron, repitan además la comparación de la sección 10 con `n_clusters=4`: ¿mejoran K-means y el jerárquico?)


Con `eps = 100 km`, prueben `min_samples = 5` y `min_samples = 20`. ¿Cuántos grupos y cuánto ruido obtienen en cada caso? ¿Por qué?

## <font color=green>C. Proyecto</font>

Si su proyecto tiene componente espacial, indiquen cuál de estos enfoques usarían y por qué:

- clustering por atributos y luego mapa;
- clustering espacial (DBSCAN con coordenadas en metros);
- mapa de calor.

Si el proyecto no tiene componente espacial, no es necesario inventarlo.